# Moon Rover MPC Testing

In [ ]:
%matplotlib ipympl

import time
import functools
import jax
import jax.numpy as jnp
import numpy as np
import tqdm
import scipy.optimize as sci_opt
import pandas as pd
import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.quartic_cost as quartic_cost
import exp_mpc.stewart_min.mp_mpl as mp_mpl

jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

In [ ]:
def load_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    """Load reference linear accelerations and angular velocities."""
    df = pd.read_csv(file_path)

    acc_keys = [f"sesmt.md.merged_frame.xyz_acc[{i}] {{m/s^2}}" for i in range(3)]
    omega_keys = [f"sesmt.md.merged_frame.ang_vel[{i}] {{rad/s}}" for i in range(3)]
    gravity_keys = [
        f"sesmt.md.merged_frame.gravity[{i}] {{m/s^2}}" for i in range(3)
    ]

    acc_ref = jnp.array(df[acc_keys])
    omega_ref = jnp.array(df[omega_keys])
    gravity_ref = jnp.array(df[gravity_keys])

    # for some reason, data collection after a lot of nonsense data
    # we grab the data after we start recognizing nonzero (x) accelerations
    # note that using direct equality is desired here (and not jnp.isclose)
    offset = jnp.argmax(acc_ref[:, 0] != 0.0)

    # we need to cancel and then add back in the gravity vector
    acc_ref = acc_ref[offset:, :] - 2 * gravity_ref[offset:, :]
    omega_ref = omega_ref[offset:, :]
    
    return acc_ref, omega_ref


In [ ]:
# load data
file_path = "/Users/jozbee/work/eng/comp/data/00_sms_drive.csv"
acc_ref, omega_ref = load_sms_references(file_path)
assert acc_ref.shape[0] == omega_ref.shape[0]
assert acc_ref.shape[1] == 3
assert omega_ref.shape[1] == 3

In [ ]:
# general setup
# num_steps = acc_ref.shape[0]
begin = 3000
# num_steps = acc_ref.shape[0] - begin
num_steps = 1000
n = 400  # horizon

In [ ]:
# cost setup

# weights = opt.ExpWeights(
#     acc=jnp.array([1e2, 1e2, 1e0]),
#     omega=jnp.array([1e2, 1e2, 1e2]),
#     alpha_acc=jnp.array([4.0]),
#     alpha_omega=jnp.array([4.0]),
# )

weights = opt.ExpWeights(
    acc=jnp.array([1e1, 1e1, 1e0]),
    alpha_acc=jnp.array([4.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    # margins=margins,
    # sizes=sizes,
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
)

In [ ]:
# train setup
get_cost = functools.partial(
    opt.get_cost,
    weights=weights,
    cost_terms=cost_terms,
    # acc_ref=acc_ref,
    # omega_ref=omega_ref,
    # state0=state0,
)


def train_step(
    state0: jax.Array,
    last_control: jax.Array,
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    **kwargs,
) -> tuple[jax.Array, jax.Array, utils.TableSol, sci_opt.OptimizeResult, float]:
    acc_ref = jnp.ravel(acc_ref)
    omega_ref = jnp.ravel(omega_ref)
    assert acc_ref.size == 3
    assert omega_ref.size == 3

    acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
    omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

    cost, cost_jac, _ = get_cost(
        acc_ref=acc_ref, omega_ref=omega_ref, state0=state0
    )
    t0 = time.time()
    opt_options = kwargs.get("opt_options", {"maxiter": 16, "maxls": 8})
    res = sci_opt.minimize(
        fun=cost,
        x0=np.array(last_control),
        method="L-BFGS-B",
        jac=cost_jac,
        options=opt_options,
    )
    t1 = time.time()
    t_tot = t1 - t0
    control = opt.Control.from_flat(jnp.array(res.x))
    state = control.get_state(state0)
    table_sol = utils.TableSol(
        x=jnp.column_stack(list(state)),
        u=jnp.column_stack(list(control)),
        stats=utils.TableStats(
            time=jnp.squeeze(t_tot),
            status=jnp.squeeze(res.status),
            cost=jnp.squeeze(res.fun),
        ),
    )
    return state[1].flatten(), control.flatten(), table_sol, res, t_tot


In [ ]:
# run
state0 = jnp.zeros(12, dtype=float)
_, last_control, _, _, _ = train_step(
    state0, jnp.zeros(6 * n, dtype=float), acc_ref[begin + 0], omega_ref[begin + 0]
)

times = []
sol_list = []
res_list = []

In [ ]:
# run
for i in tqdm.tqdm(range(num_steps)):
    state0, last_control, sol, res, t_tot = train_step(
        state0, last_control, acc_ref[begin + i], omega_ref[begin + i], opt_options={"maxiter": 2, "maxls": 4}
    )
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
trajectory = sol_list
references={
    "xyz-acceleration": jnp.array(acc_ref[begin: begin + num_steps]),
    "angular-velocity": jnp.array(omega_ref[begin: begin + num_steps]),
}

In [ ]:
acados_human_fig = viz.plot_human_trajectory(trajectory=trajectory, references=references)

In [ ]:
# acados_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory)

In [ ]:
acados_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory)

In [ ]:
# plot cost history
if True:
    costs = []
    for i, sol in enumerate(sol_list):
        cost_fun, _, _ = get_cost(
            state0=sol.x[0],
            acc_ref=jnp.tile(acc_ref[begin + i], (n, 1)),
            omega_ref=jnp.tile(omega_ref[begin + i], (n, 1)),
        )
        costs.append(cost_fun(sol.u))

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
    ax.set_title("Cost history")
    ax.set_xlabel("iter")
    ax.set_ylabel("cost")
    ax.plot(costs)
    ax.grid()
    fig.tight_layout()
    plt.show()

In [ ]:
# assert False

## animations

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
mp_mpl.call_mp_animate_trajectory(
    file_name="data/sms_acc_3d.mp4",
    trajectory=trajectory,
)

In [ ]:
mp_mpl.call_mp_animate_human_trajectory(
    file_name="data/sms_acc_human.mp4",
    trajectory=trajectory,
    references=references,
)

In [ ]:

def cost_trajectory_tile(arr: jax.Array) -> jax.Array:
    arr = arr[begin: begin + num_steps]
    arr = arr.reshape(arr.shape[0], 1, arr.shape[1])
    return jnp.tile(arr, (1, n, 1))

mp_mpl.call_mp_animate_cost_trajectory(
    file_name="data/sms_acc_cost.mp4",
    acc_refs=cost_trajectory_tile(acc_ref),
    omega_refs=cost_trajectory_tile(omega_ref),
    weights=weights,
    cost_terms=cost_terms,
    trajectory=trajectory,
)